# Lecture 30 - Working with Larger Datasets: Performance and Optimisation

## Learning Objectives

- Profile code with %timeit, cProfile, and .memory_usage()
- Replace iterative operations with vectorised Pandas code
- Use df.eval() and df.query() for faster expressions
- Process large datasets in chunks with pd.read_csv(chunksize=...)
- Parallelise tasks with concurrent.futures
- Identify when to move to databases or Spark

## Key Topics

- %timeit, cProfile, .memory_usage()
- Vectorisation vs iteration
- df.eval() and df.query()
- Chunked reading with pd.read_csv(chunksize=...)
- concurrent.futures
- Dask introduction
- When to move to database or Spark

## Profiling Code: Time and Memory

Before optimising, you must measure. **Profiling** identifies where your code actually spends time and memory, so you focus your efforts where they matter most.

- **%timeit**: IPython magic command that runs a statement many times and reports the best execution time. Use it to compare small code snippets.
- **cProfile**: built-in module that profiles an entire script and produces a detailed report of function calls and timings.
- **.memory_usage()**: Pandas method that returns memory usage per column in bytes. Use `.info(memory_usage="deep")` for an entire DataFrame.

Rule of thumb: profile first, optimise second. Don't guess where the bottleneck is — measure it.

In [ ]:
# %timeit example (run in a notebook cell, shown for reference)
# %%timeit
# result = sum(range(1000000))

# Memory profiling with Pandas
import pandas as pd
df_mem = pd.DataFrame({
    "int_col": range(100000),
    "float_col": [float(i) for i in range(100000)],
    "str_col": [f"string_{i}" for i in range(100000)]
})
print("Memory per column (bytes):")
print(df_mem.memory_usage(deep=True))
print(f"\nTotal memory: {df_mem.memory_usage(deep=True).sum() / 1024:.2f} KB")

In [ ]:
# cProfile usage example (conceptual — run in script)
import cProfile
import pstats

def slow_function():
    total = 0
    for i in range(100000):
        total += i ** 2
    return total

# Uncomment to profile:
# profiler = cProfile.Profile()
# profiler.enable()
# result = slow_function()
# profiler.disable()
# pstats.Stats(profiler).sort_stats("cumulative").print_stats(10)
print("cProfile example ready — uncomment lines above to run.")

## Vectorisation vs Iteration in Pandas

Pandas is built on NumPy, which is written in C. **Vectorised operations** (applying operations to entire columns at once) run at C speed. Iterating row-by-row with `for` loops or `df.iterrows()` forces Python-level overhead on every row, making it 100-1000x slower.

The Golden Rule of Pandas: **never iterate when you can vectorise**.

- Use `df["col"] * 2` instead of a loop over rows
- Use `df["col"].apply(my_function)` when you need element-wise custom logic
- Only fall back to `df.iterrows()` or `df.itertuples()` when vectorisation is truly impossible

If you find yourself writing a for loop over rows in Pandas, stop and ask if there is a vectorised alternative.

In [ ]:
import numpy as np
import pandas as pd
import time

df_vec = pd.DataFrame({
    "a": np.random.rand(100000),
    "b": np.random.rand(100000)
})

# Vectorised operation (fast)
start = time.time()
result_vec = df_vec["a"] * df_vec["b"] + df_vec["a"] ** 2
vec_time = time.time() - start

# Loop operation (slow)
start = time.time()
result_loop = []
for i in range(len(df_vec)):
    result_loop.append(df_vec["a"].iloc[i] * df_vec["b"].iloc[i] + df_vec["a"].iloc[i] ** 2)
loop_time = time.time() - start

print(f"Vectorised: {vec_time:.4f}s")
print(f"Loop:       {loop_time:.4f}s")
print(f"Speed-up:   {loop_time / vec_time:.0f}x")

In [ ]:
# Using .apply() as a middle ground
def my_func(row):
    return row["a"] * row["b"] + row["a"] ** 2

start = time.time()
result_apply = df_vec.apply(my_func, axis=1)
apply_time = time.time() - start
print(f".apply(): {apply_time:.4f}s  ({vec_time:.4f}s for vectorised)")

## df.eval() and df.query() for Fast Expressions

Pandas' `eval()` and `query()` use **numexpr** under the hood to evaluate expressions without creating intermediate DataFrames. This saves memory and computation.

- `df.eval("new_col = a + b * c")` — compute column expressions efficiently
- `df.query("a > 0.5 & b < 0.8")` — filter rows without building boolean masks

These methods shine on large DataFrames where intermediate arrays would consume significant memory. For small DataFrames (under 10,000 rows), the overhead is often larger than the benefit, so use them when your data is big.

In [ ]:
df_eval = pd.DataFrame(np.random.rand(1000000, 5), columns=list("abcde"))

# Traditional method
start = time.time()
df_eval["f"] = df_eval["a"] + df_eval["b"] * df_eval["c"] - df_eval["d"]
trad_time = time.time() - start

# Using eval
start = time.time()
df_eval.eval("g = a + b * c - d", inplace=True)
eval_time = time.time() - start

print(f"Traditional: {trad_time:.4f}s")
print(f"eval():      {eval_time:.4f}s")
print(f"Speed-up:    {trad_time / eval_time:.2f}x")

In [ ]:
# Using query for filtering
filtered_trad = df_eval[(df_eval["a"] > 0.5) & (df_eval["b"] < 0.3)]
filtered_query = df_eval.query("a > 0.5 & b < 0.3")

print(f"Traditional filter rows: {len(filtered_trad)}")
print(f"Query filter rows:       {len(filtered_query)}")
print("Results match:", filtered_trad.equals(filtered_query))

## Chunked Reading with pd.read_csv(chunksize=...)

When a dataset is too large to fit in memory, you can process it in **chunks**. `pd.read_csv(chunksize=N)` returns an iterator that yields DataFrames of N rows each. You process each chunk, aggregate the results, and combine them at the end.

This approach works for:
- Computing summary statistics across the entire dataset
- Filtering rows and writing to a smaller output file
- Feature extraction that is row-independent

When even chunked processing is too slow or memory-intensive, it is time to consider out-of-core tools like Dask, a database, or Spark.

In [ ]:
# Simulate chunked processing on a large dataset
import tempfile, os

# Create a large CSV
large_df = pd.DataFrame(np.random.rand(100000, 5), columns=list("abcde"))
large_df.to_csv("/tmp/large_data.csv", index=False)
total_rows = 0
col_sums = np.zeros(5)

for chunk in pd.read_csv("/tmp/large_data.csv", chunksize=10000):
    total_rows += len(chunk)
    col_sums += chunk.sum(numeric_only=True).values

print(f"Processed {total_rows} rows in chunks")
print(f"Column means: {col_sums / total_rows}")

os.remove("/tmp/large_data.csv")

## Parallel Processing with concurrent.futures

Modern CPUs have multiple cores, but standard Python code only uses one due to the Global Interpreter Lock (GIL). For **CPU-bound** tasks, you can bypass the GIL using `multiprocessing`. For **I/O-bound** tasks (file reads, API calls), threading works well.

The `concurrent.futures` module provides a clean, high-level interface:
- `ThreadPoolExecutor` for I/O-bound tasks
- `ProcessPoolExecutor` for CPU-bound tasks

Both use a `.map()` or `.submit()` pattern to distribute work across workers. The overhead of process/thread creation means parallelism only helps when each unit of work is substantial (at least 0.1s).

In [ ]:
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import time

def compute_square(x):
    return sum(i ** 2 for i in range(x))

numbers = [20000, 30000, 25000, 40000, 35000, 15000, 50000, 45000]

# Sequential
start = time.time()
seq_results = [compute_square(n) for n in numbers]
seq_time = time.time() - start
print(f"Sequential: {seq_time:.3f}s")

# Parallel with ProcessPoolExecutor
start = time.time()
with ProcessPoolExecutor(max_workers=4) as executor:
    par_results = list(executor.map(compute_square, numbers))
par_time = time.time() - start
print(f"Parallel (4 workers): {par_time:.3f}s")
print(f"Speed-up: {seq_time / par_time:.1f}x")

## When to Move to a Database or Spark

Pandas is fantastic for datasets that fit in memory (up to ~10-20 GB on a typical laptop). Beyond that, consider:

1. **Dask**: a drop-in replacement for Pandas that works on larger-than-memory data and distributes across cores or clusters. It uses lazy evaluation and a task scheduler.
2. **SQL database**: for structured, relational data, a database (PostgreSQL, DuckDB) can filter and aggregate far more efficiently than Pandas, using indexes and query optimisation.
3. **Apache Spark**: for truly large-scale data (terabytes+), Spark distributes computation across a cluster. PySpark provides a Pandas-like API.

Rule of thumb: if your data fits in RAM, use Pandas. If it doesn't, try chunks. If chunks are too slow, try Dask or a database. If the data is huge (multi-TB), use Spark.

In [ ]:
# Dask example (conceptual — requires dask installed)
# import dask.dataframe as dd
# ddf = dd.read_csv("huge_dataset.csv")
# result = ddf.groupby("category").mean().compute()
# print(result)
print("Dask usage pattern shown (uncomment to run with dask installed).")

# SQL alternative example
import sqlite3
conn = sqlite3.connect("/tmp/example.db")
df_sql = pd.DataFrame(np.random.rand(1000, 3), columns=list("xyz"))
df_sql.to_sql("mydata", conn, if_exists="replace", index=False)

query_result = pd.read_sql_query("SELECT AVG(x), AVG(y), AVG(z) FROM mydata", conn)
print("\nSQL aggregation result:")
print(query_result)
conn.close()
os.remove("/tmp/example.db")

## Data Science Connection

Performance and scalability are what separate hobbyist scripts from production data pipelines. As datasets grow from megabytes to gigabytes to terabytes, the techniques in this lecture — profiling, vectorisation, chunking, and parallelism — become essential. Every major tech company deals with data at scale; knowing when and how to optimise is a core skill for senior data scientists and data engineers. Start with Pandas, scale up with Dask, and reach for Spark when you must.